MovieLens Rating Prediction Notebook

This notebook runs faster on a GPU runtime. To enable it, go to Edit > Notebook Settings > Hardware Accelerator > GPU.


## Setup

In [ ]:
import torch

print(torch.__version__)


2.4.0+cu121


In [ ]:
# Install required packages
import os

os.environ['TORCH'] = torch.__version__
!pip install -q torch-scatter -f https://pytorch-geometric.com/whl/torch-${TORCH}.html
!pip install pyg-lib -f https://data.pyg.org/whl/nightly/torch-${TORCH}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git

!pip install sentence_transformers
!pip3 install fuzzywuzzy[speedup]
!pip install captum

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 67.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/nightly/torch-2.4.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.4 MB/s eta 0:00:00
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-9sqjx7uj
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-9sqjx7uj
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit 241a8c3d018636c116fd1fd7fa2ab9ff3925531e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for torch-geometric: filename=torch_geometric-2.6.0-py3-none-any.whl size=1128957 sha256=34bf7b3ffc2216c7acf23ebe83ed2d25282c1892ae230ee683322594208b1010
  Stored in directory: /tmp/pip-ephem-wheel-cache-p8k1dk0a/wheels/d3/78/eb/9e26525b948d19533f1688fb6c209cec8a0ba793d39b49ae8f

## Data Ingestion

In [ ]:
from torch_geometric.data import download_url, extract_zip
import pandas as pd

dataset_name = 'ml-latest-small'

url = f'https://files.grouplens.org/datasets/movielens/{dataset_name}.zip'
extract_zip(download_url(url, '.'), '.')

movies_path = f'./{dataset_name}/movies.csv'
ratings_path = f'./{dataset_name}/ratings.csv'

Extracting ./ml-latest-small.zip


In [ ]:
# Load the entire ratings dataframe into memory:
ratings_df = pd.read_csv(ratings_path)[["userId", "movieId", "rating"]]

# Load the entire movie dataframe into memory:
movies_df = pd.read_csv(movies_path, index_col='movieId')

print('movies.csv:')
print('===========')
print(movies_df[["genres", "title"]].head())
print(f"Number of movies: {len(movies_df)}")
print()
print('ratings.csv:')
print('============')
print(ratings_df[["userId", "movieId", "rating"]].head())
print(f"Number of ratings: {len(ratings_df)}")
print()

movies.csv:
                                              genres  \
movieId                                                
1        Adventure|Animation|Children|Comedy|Fantasy   
2                         Adventure|Children|Fantasy   
3                                     Comedy|Romance   
4                               Comedy|Drama|Romance   
5                                             Comedy   

                                      title  
movieId                                      
1                          Toy Story (1995)  
2                            Jumanji (1995)  
3                   Grumpier Old Men (1995)  
4                  Waiting to Exhale (1995)  
5        Father of the Bride Part II (1995)  
Number of movies: 9742

ratings.csv:
   userId  movieId  rating
0       1        1     4.0
1       1        3     4.0
2       1        6     4.0
3       1       47     5.0
4       1       50     5.0
Number of ratings: 100836



In [ ]:
from fuzzywuzzy import fuzz

# Specify your userId
our_user_id = ratings_df['userId'].max() + 1

print('Most rated movies:')
print('==================')
most_rated_movies = ratings_df['movieId'].value_counts().head(10)
print(movies_df.loc[most_rated_movies.index][["title"]])

# Initialize your rating list
ratings = []

Most rated movies:
                                             title
movieId                                           
356                            Forrest Gump (1994)
318               Shawshank Redemption, The (1994)
296                            Pulp Fiction (1994)
593               Silence of the Lambs, The (1991)
2571                            Matrix, The (1999)
260      Star Wars: Episode IV - A New Hope (1977)
480                           Jurassic Park (1993)
110                              Braveheart (1995)
589              Terminator 2: Judgment Day (1991)
527                        Schindler's List (1993)


In [ ]:
# Add your ratings here:
num_ratings = 5

while len(ratings) < num_ratings:
    print(f'Select the {len(ratings) + 1}. movie:')
    print('=====================================')
    movie = input('Please enter the movie title: ')
    movies_df['title_score'] = movies_df['title'].apply(lambda x: fuzz.ratio(x, movie))
    print(movies_df.sort_values('title_score', ascending=False)[['title']].head(5))
    movie_id = input('Please enter the movie id: ')
    if not movie_id:
        continue
    movie_id = int(movie_id)
    rating = float(input('Please enter your rating: '))
    if not rating:
        continue
    assert 0 <= rating <= 5
    ratings.append({'movieId': movie_id, 'rating': rating, 'userId': our_user_id})
    print()

Select the 1. movie:
Please enter the movie title: maze runner
                            title
movieId                          
114180    Maze Runner, The (2014)
541           Blade Runner (1982)
140237          The Runner (2015)
176371   Blade Runner 2049 (2017)
105351       Runner Runner (2013)
Please enter the movie id: 114180    
Please enter your rating: 5

Select the 2. movie:
Please enter the movie title: 176371   
                       title
movieId                     
5472             1776 (1972)
5300     3:10 to Yuma (1957)
54997    3:10 to Yuma (2007)
110286        13 Sins (2014)
1260                M (1931)
Please enter the movie id: 54997    
Please enter your rating: 4

Select the 3. movie:
Please enter the movie title: samurai
                            title
movieId                          
139130        Afro Samurai (2007)
7143     Last Samurai, The (2003)
7177                 Osama (2003)
942                  Laura (1944)
42004         Transamerica (2005)
Pleas

In [ ]:
# Add your ratings to the rating dataframe
ratings_df = pd.concat([ratings_df, pd.DataFrame.from_records(ratings)])

In [ ]:
# Select our userId
our_user_id = ratings_df['userId'].max() + 1

# Load the IMDB ratings:
imdb_rating_path = f'./imdb_ratings.csv'
imdb_ratings_df = pd.read_csv(imdb_rating_path)
imdb_ratings_df.columns = imdb_ratings_df.columns.str.strip().str.lower()

# The IMDB movie titles / ids do not match the movie titles /ids in the movielens dataframes
# so we need to map them:
imdb_ratings_df['title'] = imdb_ratings_df['title'] + ' (' + imdb_ratings_df['year'].astype(str) + ')'
imdb_ratings_df['title'] = imdb_ratings_df['title'].str.strip()
movies_df['title'] = movies_df['title'].str.strip()
imdb_ratings_df = imdb_ratings_df.merge(movies_df['title'].reset_index(), on='title', how='inner', )

# The ratings are on a scale from 1 to 10, so we need to transform them to a scale from 0 to 5:
imdb_ratings_df['rating'] = (imdb_ratings_df['your rating'] / 2).astype(int)

# Your ratings that we are going to use:
print('Your IMDB ratings:')
print('==================')
print(imdb_ratings_df[['title', 'rating']].head(10))

# Finally, we can add the ratings to the ratings data frame:
imdb_ratings_df['userId'] = our_user_id
ratings_df = pd.concat([ratings_df, imdb_ratings_df[['movieId', 'rating', 'userId']]])

FileNotFoundError: [Errno 2] No such file or directory: './imdb_ratings.csv'

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# One-hot encode the genres:
genres = movies_df['genres'].str.get_dummies('|').values
genres = torch.from_numpy(genres).to(torch.float)

# Load the pre-trained sentence transformer model and encode the movie titles:
model = SentenceTransformer('all-MiniLM-L6-v2')
with torch.no_grad():
    titles = model.encode(movies_df['title'].tolist(), convert_to_tensor=True, show_progress_bar=True)
    titles = titles.cpu()

# Concatenate the genres and title features:
movie_features = torch.cat([genres, titles], dim=-1)

# We don't have user features, which is why we use an identity matrix
user_features = torch.eye(len(ratings_df['userId'].unique()))


/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

In [ ]:
# Create a mapping from the userId to a unique consecutive value in the range [0, num_users]:
unique_user_id = ratings_df['userId'].unique()
unique_user_id = pd.DataFrame(data={
    'userId': unique_user_id,
    'mappedUserId': pd.RangeIndex(len(unique_user_id))
    })
print("Mapping of user IDs to consecutive values:")
print("==========================================")
print(unique_user_id.head())
print()

# Create a mapping from the movieId to a unique consecutive value in the range [0, num_movies]:
unique_movie_id = ratings_df['movieId'].unique()
unique_movie_id = pd.DataFrame(data={
    'movieId': unique_movie_id,
    'mappedMovieId': pd.RangeIndex(len(unique_movie_id))
    })
print("Mapping of movie IDs to consecutive values:")
print("===========================================")
print(unique_movie_id.head())
print()

# Merge the mappings with the original data frame:
ratings_df = ratings_df.merge(unique_user_id, on='userId')
ratings_df = ratings_df.merge(unique_movie_id, on='movieId')

# With this, we are ready to create the edge_index representation in COO format
# following the PyTorch Geometric semantics:
edge_index = torch.stack([
    torch.tensor(ratings_df['mappedUserId'].values),
    torch.tensor(ratings_df['mappedMovieId'].values)]
    , dim=0)

assert edge_index.shape == (2, len(ratings_df))

print("Final edge indices pointing from users to movies:")
print("================================================")
print(edge_index[:, :10])

Mapping of user IDs to consecutive values:
   userId  mappedUserId
0       1             0
1       2             1
2       3             2
3       4             3
4       5             4

Mapping of movie IDs to consecutive values:
   movieId  mappedMovieId
0        1              0
1        3              1
2        6              2
3       47              3
4       50              4

Final edge indices pointing from users to movies:
tensor([[ 0,  4,  6, 14, 16, 17, 18, 20, 26, 30],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]])


In [ ]:
import torch_geometric.transforms as T
from torch_geometric.data import HeteroData

# Create the heterogeneous graph data object:
data = HeteroData()

# Add the user nodes:
data['user'].x = user_features  # [num_users, num_features_users]

# Add the movie nodes:
data['movie'].x = movie_features  # [num_movies, num_features_movies]

# Add the rating edges:
data['user', 'rates', 'movie'].edge_index = edge_index  # [2, num_ratings]

# Add the rating labels:
rating = torch.from_numpy(ratings_df['rating'].values).to(torch.float)
data['user', 'rates', 'movie'].edge_label = rating  # [num_ratings]

# We also need to make sure to add the reverse edges from movies to users
# in order to let a GNN be able to pass messages in both directions.
# We can leverage the `T.ToUndirected()` transform for this from PyG:
data = T.ToUndirected()(data)

# With the above transformation we also got reversed labels for the edges.
# We are going to remove them:
del data['movie', 'rev_rates', 'user'].edge_label

assert data['user'].num_nodes == len(unique_user_id)
assert data['user', 'rates', 'movie'].num_edges == len(ratings_df)
assert data['movie'].num_features == 404

data

HeteroData(
  user={ x=[611, 611] },
  movie={ x=[9742, 404] },
  (user, rates, movie)={
    edge_index=[2, 100841],
    edge_label=[100841],
  },
  (movie, rev_rates, user)={ edge_index=[2, 100841] }
)

In [ ]:
train_data, val_data, test_data = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    neg_sampling_ratio=0.0,
    edge_types=[('user', 'rates', 'movie')],
    rev_edge_types=[('movie', 'rev_rates', 'user')],
)(data)
train_data, val_data

(HeteroData(
   user={ x=[611, 611] },
   movie={ x=[9742, 404] },
   (user, rates, movie)={
     edge_index=[2, 80673],
     edge_label=[80673],
     edge_label_index=[2, 80673],
   },
   (movie, rev_rates, user)={ edge_index=[2, 80673] }
 ),
 HeteroData(
   user={ x=[611, 611] },
   movie={ x=[9742, 404] },
   (user, rates, movie)={
     edge_index=[2, 80673],
     edge_label=[10084],
     edge_label_index=[2, 10084],
   },
   (movie, rev_rates, user)={ edge_index=[2, 80673] }
 ))

In [ ]:
import torch.nn.functional as F

import torch_scatter
from torch_geometric.nn.conv import MessagePassing

class GraphSage(MessagePassing):

    def __init__(self, in_channels, out_channels, normalize = True,
                 bias = False, **kwargs):
        super(GraphSage, self).__init__(**kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.normalize = normalize

        self.lin_l = None
        self.lin_r = None

        ############################################################################
        # TODO: Your code here!
        # Define the layers needed for the message and update functions below.
        # self.lin_l is the linear transformation that you apply to embedding
        #            for central node.
        # self.lin_r is the linear transformation that you apply to aggregated
        #            message from neighbors.
        # Our implementation is ~2 lines, but don't worry if you deviate from this.



        ############################################################################

        self.reset_parameters()

    def reset_parameters(self):
        self.lin_l.reset_parameters()
        self.lin_r.reset_parameters()

    def forward(self, x, edge_index, size = None):
        """"""

        out = None

        ############################################################################
        # TODO: Your code here!
        # Implement message passing, as well as any post-processing (our update rule).
        # 1. First call propagate function to conduct the message passing.
        #    1.1 See there for more information:
        #        https://pytorch-geometric.readthedocs.io/en/latest/notes/create_gnn.html
        #    1.2 We use the same representations for central (x_central) and
        #        neighbor (x_neighbor) nodes, which means you'll pass x=(x, x)
        #        to propagate.
        # 2. Update our node embedding with skip connection.
        # 3. If normalize is set, do L-2 normalization (defined in
        #    torch.nn.functional)
        # Our implementation is ~5 lines, but don't worry if you deviate from this.



        ############################################################################

        return out

    def message(self, x_j):

        out = None

        ############################################################################
        # TODO: Your code here!
        # Implement your message function here.
        # Our implementation is ~1 lines, but don't worry if you deviate from this.


        # x_j.shape == [num_edges, num_node_features]
        #  Basically, for every edge on the graph there is a message that will be
        #  transferred by it. This message has some representation (hidden state).


        ############################################################################

        return out

    def aggregate(self, inputs, index, dim_size = None):

        out = None

        # The axis along which to index number of nodes.
        node_dim = self.node_dim

        ############################################################################
        # TODO: Your code here!
        # Implement your aggregate function here.
        # See here as how to use torch_scatter.scatter:
        # https://pytorch-scatter.readthedocs.io/en/latest/functions/scatter.html#torch_scatter.scatter
        # Our implementation is ~1 lines, but don't worry if you deviate from this.


        # 1. input.shape == [num_edges, num_node_features]
        #       (basically the output of self.message)

        # 2. index.shape == [num_edges]
        #       (destination node of each edge, basically takes values between [0, ..., num_nodes - 1])

        # 3. self.node_dim == -2
        #       (the dimension of the nodes in the `inputs` tensor. It's -2 by default in the constructor
        #       of MessagePassing (https://pytorch-geometric.readthedocs.io/en/latest/_modules/torch_geometric/nn/conv/message_passing.html#MessagePassing)

        # 4. dim_size == index.max() + 1,
        #       it will be the size of the output, which will be the number of nodes as we want
        #       to compute the aggregated message for every node

        # aggregate the message for every node
        #  The shape of the output will be: [num_nodes, num_features]
        #  (i.e. the aggregated message of every node)

        ############################################################################

        return out


## Graph neural network (GNN) model
We are now ready to define our GNN model. We are going to use a simple GNN model with
two message passing layers for the encoding of the user and movie nodes.
Additionally, we are going to use a decoder to predict the rating for the encoded
user-movie combination.

<font color='red'>You may choose your own GNN layers implemented above, or you can use the provided GNN layers, such as GAT, GCN, etc., from the PyTorch Geometric library</font>

In [ ]:
from torch_geometric.nn import SAGEConv, to_hetero, GCNConv

class GNNEncoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

class GNNEncoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)


        ############################################################################
        # TODO: Your code here!
        # Initialize two Graph convolution layers.
        # The first layer should map from the input dimension to hidden_channels.
        # The second layer should map from hidden_channels to out_channels.
        #
        # Hint 1: Use SAGEConv or any other conv operators from torch_geometric.nn
        # Hint 2: The input dimension is unknown at initialization, so use (-1, -1)
        #         as the input size for both layers.
        #
        # Your implementation should be 2 lines of code.


        ############################################################################

    def forward(self, x, edge_index):
      x = self.conv1(x, edge_index).relu()
      x = self.conv2(x, edge_index)


        ############################################################################
        # TODO: Your code here!
        # Implement the forward pass of the GNN encoder.
        # 1. Apply the first Graph conv layer (self.conv1) to the input.
        # 2. Apply a ReLU activation function to the output of the first layer.
        # 3. Apply the second Graph conv layer (self.conv2) to the result.
        # 4. Return the final output.
        #
        # Hint 1: Use the conv1 and conv2 layers you defined in __init__
        # Hint 2: You can chain operations using method chaining, e.g., .relu()
        #
        # Your implementation should be 2-3 lines of code.


        ############################################################################

      return x

class EdgeDecoder(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.lin1 = torch.nn.Linear(2 * hidden_channels, hidden_channels)
        self.lin2 = torch.nn.Linear(hidden_channels, 1)

    def forward(self, z_dict, edge_label_index):
        row, col = edge_label_index
        z = torch.cat([z_dict['user'][row], z_dict['movie'][col]], dim=-1)

        z = self.lin1(z).relu()
        z = self.lin2(z)
        return z.view(-1)


class Model(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.encoder = GNNEncoder(hidden_channels, hidden_channels)
        self.encoder = to_hetero(self.encoder, data.metadata(), aggr='sum')
        self.decoder = EdgeDecoder(hidden_channels)

    def forward(self, x_dict, edge_index_dict, edge_label_index):
        z_dict = self.encoder(x_dict, edge_index_dict)
        return self.decoder(z_dict, edge_label_index)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Model(hidden_channels=32).to(device)

print(model)

Model(
  (encoder): GraphModule(
    (conv1): ModuleDict(
      (user__rates__movie): SAGEConv((-1, -1), 32, aggr=mean)
      (movie__rev_rates__user): SAGEConv((-1, -1), 32, aggr=mean)
    )
    (conv2): ModuleDict(
      (user__rates__movie): SAGEConv((-1, -1), 32, aggr=mean)
      (movie__rev_rates__user): SAGEConv((-1, -1), 32, aggr=mean)
    )
  )
  (decoder): EdgeDecoder(
    (lin1): Linear(in_features=64, out_features=32, bias=True)
    (lin2): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [ ]:
import torch.nn.functional as F

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def train():
    model.train()
    optimizer.zero_grad()
    pred = model(train_data.x_dict, train_data.edge_index_dict,
                 train_data['user', 'movie'].edge_label_index)
    target = train_data['user', 'movie'].edge_label
    loss = F.mse_loss(pred, target)
    loss.backward()
    optimizer.step()
    return float(loss)

@torch.no_grad()
def test(data):
    data = data.to(device)
    model.eval()
    pred = model(data.x_dict, data.edge_index_dict,
                 data['user', 'movie'].edge_label_index)
    pred = pred.clamp(min=0, max=5)
    target = data['user', 'movie'].edge_label.float()
    rmse = F.mse_loss(pred, target).sqrt()
    return float(rmse)


for epoch in range(1, 301):
    train_data = train_data.to(device)
    loss = train()
    train_rmse = test(train_data)
    val_rmse = test(val_data)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train: {train_rmse:.4f}, '
          f'Val: {val_rmse:.4f}')

Epoch: 001, Loss: 14.0916, Train: 3.6491, Val: 3.6363
Epoch: 002, Loss: 13.3164, Train: 3.5103, Val: 3.4986
Epoch: 003, Loss: 12.3221, Train: 3.2746, Val: 3.2646
Epoch: 004, Loss: 10.7230, Train: 2.8726, Val: 2.8656
Epoch: 005, Loss: 8.2520, Train: 2.2243, Val: 2.2227
Epoch: 006, Loss: 4.9476, Train: 1.3056, Val: 1.3133
Epoch: 007, Loss: 1.7047, Train: 1.4634, Val: 1.4615
Epoch: 008, Loss: 2.1498, Train: 1.8027, Val: 1.7949
Epoch: 009, Loss: 4.5112, Train: 1.6603, Val: 1.6553
Epoch: 010, Loss: 2.9046, Train: 1.1582, Val: 1.1636
Epoch: 011, Loss: 1.3414, Train: 1.0710, Val: 1.0820
Epoch: 012, Loss: 1.1470, Train: 1.2925, Val: 1.3011
Epoch: 013, Loss: 1.6704, Train: 1.4749, Val: 1.4813
Epoch: 014, Loss: 2.1753, Train: 1.5354, Val: 1.5411
Epoch: 015, Loss: 2.3574, Train: 1.4827, Val: 1.4891
Epoch: 016, Loss: 2.1985, Train: 1.3441, Val: 1.3524
Epoch: 017, Loss: 1.8066, Train: 1.1668, Val: 1.1776
Epoch: 018, Loss: 1.3613, Train: 1.0406, Val: 1.0528
Epoch: 019, Loss: 1.0829, Train: 1.0655, V

In [ ]:
with torch.no_grad():
    test_data = test_data.to(device)
    pred = model(test_data.x_dict, test_data.edge_index_dict,
                 test_data['user', 'movie'].edge_label_index)
    pred = pred.clamp(min=0, max=5)
    target = test_data['user', 'movie'].edge_label.float()
    rmse = F.mse_loss(pred, target).sqrt()
    print(f'Test RMSE: {rmse:.4f}')

userId = test_data['user', 'movie'].edge_label_index[0].cpu().numpy()
movieId = test_data['user', 'movie'].edge_label_index[1].cpu().numpy()
pred = pred.cpu().numpy()
target = target.cpu().numpy()

print(pd.DataFrame({'userId': userId, 'movieId': movieId, 'rating': pred, 'target': target}))

Test RMSE: 0.9121
       userId  movieId    rating  target
0         350     6044  2.735110     3.5
1         579     2046  3.830889     5.0
2         607       36  3.982919     3.0
3         143     1642  3.680841     3.5
4         554     4087  3.322887     2.0
...       ...      ...       ...     ...
10079     140       13  3.089428     3.0
10080     345      477  3.405957     2.5
10081     447     1606  3.246300     4.0
10082     533     3113  3.850991     3.0
10083     216     1030  3.223839     3.0

[10084 rows x 4 columns]


In [ ]:
# Your mappedUserId
mapped_user_id = unique_user_id[unique_user_id['userId'] == our_user_id]['mappedUserId'].values[0]

# Select movies that you haven't seen before
movies_rated = ratings_df[ratings_df['mappedUserId'] == mapped_user_id]
movies_not_rated = movies_df[~movies_df.index.isin(movies_rated['movieId'])]
movies_not_rated = movies_not_rated.merge(unique_movie_id, on='movieId')
movie = movies_not_rated.sample(1)

print(f"The movie we want to predict a raiting for is:  {movie['title'].item()}")

The movie we want to predict a raiting for is:  Golden Compass, The (2007)


In [ ]:
# Create new `edge_label_index` between the user and the movie
edge_label_index = torch.tensor([
    mapped_user_id,
    movie.mappedMovieId.item()])


with torch.no_grad():
    test_data.to(device)
    pred = model(test_data.x_dict, test_data.edge_index_dict, edge_label_index)
    pred = pred.clamp(min=0, max=5).detach().cpu().numpy()

In [ ]:
pred.item()

3.0908682346343994

In [ ]:
from torch_geometric.explain import Explainer, CaptumExplainer

explainer = Explainer(
    model=model,
    algorithm=CaptumExplainer('IntegratedGradients'),
    explanation_type='model',
    model_config=dict(
        mode='regression',
        task_level='edge',
        return_type='raw',
    ),
    node_mask_type=None,
    edge_mask_type='object',
)

explanation = explainer(
    test_data.x_dict, test_data.edge_index_dict, index=0,
    edge_label_index=edge_label_index).cpu().detach()
explanation

HeteroExplanation(
  prediction=[1],
  target=[1],
  index=[1],
  edge_label_index=[2],
  user={ x=[611, 611] },
  movie={ x=[9742, 404] },
  (user, rates, movie)={
    edge_mask=[90757],
    edge_index=[2, 90757],
  },
  (movie, rev_rates, user)={
    edge_mask=[90757],
    edge_index=[2, 90757],
  }
)

In [ ]:
# User to movie link + attribution
user_to_movie = explanation['user', 'movie'].edge_index.numpy().T
user_to_movie_attr = explanation['user', 'movie'].edge_mask.numpy().T
user_to_movie_df = pd.DataFrame(
    np.hstack([user_to_movie, user_to_movie_attr.reshape(-1,1)]),
    columns = ['mappedUserId', 'mappedMovieId', 'attr']
)

# Movie to user link + attribution
movie_to_user = explanation['movie', 'user'].edge_index.numpy().T
movie_to_user_attr = explanation[ 'movie', 'user'].edge_mask.numpy().T
movie_to_user_df = pd.DataFrame(
    np.hstack([movie_to_user, movie_to_user_attr.reshape(-1,1)]),
    columns = ['mappedMovieId', 'mappedUserId','attr']
)
explanation_df = pd.concat([user_to_movie_df, movie_to_user_df])
explanation_df[["mappedUserId", "mappedMovieId"]] = explanation_df[["mappedUserId", "mappedMovieId"]].astype(int)

print(f"Attribtion for all edges towards prediction of movie rating of movie:\n {movie['title'].item()}")
print("==========================================================================================")
print(explanation_df.sort_values(by='attr'))

Attribtion for all edges towards prediction of movie rating of movie:
 Golden Compass, The (2007)
       mappedUserId  mappedMovieId      attr
3143            381           2059 -0.017399
11123           231           2059 -0.004600
48106           610           8911 -0.004228
50571           150           5021 -0.001722
36927           491           5021 -0.001632
...             ...            ...       ...
45752           304           2059  0.017228
58678           610           5021  0.055168
65443           610           2651  0.075357
48106           610           8911  0.088929
57621           610             20  0.094208

[181514 rows x 3 columns]


In [ ]:
# Select links that connect to our user
explanation_df = explanation_df[explanation_df['mappedUserId'] == mapped_user_id]

# We group the attribution scores by movie
explanation_df = explanation_df.groupby('mappedMovieId').sum()

# Merge with movies_df to receive title
# But first, we need to add the original id
explanation_df = explanation_df.merge(unique_movie_id, on='mappedMovieId')
explanation_df = explanation_df.merge(movies_df, on='movieId')

pd.options.display.float_format = "{:,.9f}".format

print("Top movies that influenced the prediction:")
print("==============================================")
print(explanation_df.sort_values(by='attr', ascending=False, key= lambda x: abs(x))[['title', 'attr']].head())

Top movies that influenced the prediction:
                          title        attr
0           Forrest Gump (1994) 0.094178511
3               Fortress (1992) 0.084700197
1  Home for the Holidays (1995) 0.074337982
2           House Arrest (1996) 0.053983791


Find more on the official PyTorch Geometric website [here](https://www.pyg.org/).